# 01 - Exploratory Data Analysis: Heart Disease (UCI Cleveland)

**Goal:** understand the data before modelling. Run this notebook top to bottom after `python src/download_data.py`.
Figures are saved to `screenshots/eda/` so you can reuse them in the report.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

DATA = Path("../data/heart_clean.csv")      # notebook lives in notebooks/
FIG_DIR = Path("../screenshots/eda")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

df = pd.read_csv(DATA)
print("Shape:", df.shape)
df.head()

## 1. Data dictionary

| Column | Meaning | Type |
|---|---|---|
| age | Age in years | numeric |
| sex | 1 = male, 0 = female | binary |
| cp | Chest pain type (1-4) | categorical |
| trestbps | Resting blood pressure (mm Hg) | numeric |
| chol | Serum cholesterol (mg/dl) | numeric |
| fbs | Fasting blood sugar > 120 mg/dl (1 = yes) | binary |
| restecg | Resting ECG result (0-2) | categorical |
| thalach | Maximum heart rate achieved | numeric |
| exang | Exercise-induced angina (1 = yes) | binary |
| oldpeak | ST depression induced by exercise | numeric |
| slope | Slope of peak exercise ST segment (1-3) | categorical |
| ca | Number of major vessels coloured by fluoroscopy (0-3) | categorical/count |
| thal | Thalassemia: 3 = normal, 6 = fixed defect, 7 = reversible defect | categorical |
| target | 0 = no disease, 1 = disease present | **label** |

In [ ]:
df.info()
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
df.describe().T

## 2. Class balance
Is the target roughly balanced? This decides whether accuracy alone is a fair metric.

In [ ]:
counts = df["target"].value_counts().sort_index()
pct = (counts / len(df) * 100).round(1)
print(pd.DataFrame({"count": counts, "percent": pct}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="target", ax=ax)
ax.set_xticklabels(["No disease (0)", "Disease (1)"])
ax.set_title("Class balance")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
save("class_balance")
plt.show()

**Takeaway (write in your own words):** TODO - is the dataset balanced? What does that mean for the metrics you will report?

## 3. Numeric feature distributions

In [ ]:
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(col)
axes.ravel()[-1].axis("off")
fig.suptitle("Distributions of numeric features", y=1.02, fontsize=14)
save("numeric_histograms")
plt.show()

**Takeaway (write in your own words):** TODO - any skew, outliers (look at `chol` and `oldpeak`), or odd ranges?

In [ ]:
# Same features split by target - which ones separate the classes?
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x="target", y=col, ax=ax)
    ax.set_title(f"{col} by target")
axes.ravel()[-1].axis("off")
save("numeric_by_target")
plt.show()

## 4. Correlation heatmap

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.8})
plt.title("Correlation heatmap")
save("correlation_heatmap")
plt.show()

print("Correlation with target (sorted):")
print(corr["target"].drop("target").sort_values(ascending=False).round(2))

**Takeaway (write in your own words):** TODO - which features correlate most with the target? Any pair of features strongly correlated with each other?

## 5. Categorical features vs target

In [ ]:
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(data=df, x=col, hue="target", ax=ax)
    ax.set_title(f"{col} vs target")
    ax.legend(title="target", loc="upper right")
save("categorical_vs_target")
plt.show()

**Takeaway (write in your own words):** TODO - which categories show a clear difference between disease and no disease?

## 6. Your own plots (this is what makes your submission different)

Add 1-2 plots of your choice. Ideas, pick what *you* find interesting:
- Disease rate by chest pain type or by age band
- Age vs max heart rate (`thalach`) coloured by target
- A pairplot of 3-4 key features
- Missing-value pattern and how your cleaning choice affects the data

Add a short caption under each.

In [ ]:
# TODO: your own plot(s) here. Call save("your_plot_name") before plt.show()

## 7. EDA summary (copy into the report)

- **Dataset:** TODO (rows, features, source)
- **Cleaning:** TODO (how you handled missing `ca` / `thal`, and why)
- **Class balance:** TODO
- **Key patterns:** TODO (2-3 bullets)
- **Implications for modelling:** TODO (scaling needed? encode which columns? which metric matters most?)